In [1]:
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind, chi2_contingency

Before running below cell, ensure you have added the dataset path correctly based on your environement

In [2]:
# Load the dataset
df = pd.read_csv('/kaggle/input/stroke-prediction-dataset/healthcare-dataset-stroke-data.csv')  

df.drop(columns=['id'], inplace=True)

print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nColumn names and data types:")
print(df.dtypes)
print("\nClass distribution for 'stroke':")
print(df['stroke'].value_counts())
print(df['stroke'].value_counts(normalize=True) * 100)

Dataset Shape: (5110, 11)

First 5 rows:
   gender   age  hypertension  heart_disease ever_married      work_type  \
0    Male  67.0             0              1          Yes        Private   
1  Female  61.0             0              0          Yes  Self-employed   
2    Male  80.0             0              1          Yes        Private   
3  Female  49.0             0              0          Yes        Private   
4  Female  79.0             1              0          Yes  Self-employed   

  Residence_type  avg_glucose_level   bmi   smoking_status  stroke  
0          Urban             228.69  36.6  formerly smoked       1  
1          Rural             202.21   NaN     never smoked       1  
2          Rural             105.92  32.5     never smoked       1  
3          Urban             171.23  34.4           smokes       1  
4          Rural             174.12  24.0     never smoked       1  

Column names and data types:
gender                object
age                  float64


/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


# bmi missing value handling

In [3]:

df['bmi'] = df['bmi'].fillna(df['bmi'].mean())

# Ensure correct data types for categorical variables
categorical_cols = ['gender', 'hypertension', 'heart_disease', 'ever_married', 
                    'work_type', 'Residence_type', 'smoking_status']
for col in categorical_cols:
    df[col] = df[col].astype('category')

# Separate the dataset into two groups: Stroke vs. No Stroke
df_stroke = df[df['stroke'] == 1]
df_no_stroke = df[df['stroke'] == 0]

# drop 'other' from gender as 1 row only. doen't affect model
df = df[df['gender'].isin(['Male', 'Female'])].copy()

In [4]:
df.shape

(5109, 11)

## t-test for continuous variables

In [5]:
# List of continuous variables
continuous_vars = ['age', 'avg_glucose_level', 'bmi']

print("--- Analysis of Continuous Variables ---")
print("Format: Mean ± Standard Deviation")
print("")

results_continuous = []

for var in continuous_vars:
    # Calculate overall stats
    overall_mean = df[var].mean()
    overall_std = df[var].std()
    
    # Calculate stats for each group
    mean_stroke = df_stroke[var].mean()
    std_stroke = df_stroke[var].std()
    mean_no_stroke = df_no_stroke[var].mean()
    std_no_stroke = df_no_stroke[var].std()
    
    # Perform independent t-test
    t_stat, p_value = ttest_ind(df_stroke[var], df_no_stroke[var], nan_policy='omit')
    
    # Store results
    results_continuous.append({
        'Variable': var,
        'Overall': f"{overall_mean:.2f} ± {overall_std:.2f}",
        'No_Stroke': f"{mean_no_stroke:.2f} ± {std_no_stroke:.2f}",
        'Stroke': f"{mean_stroke:.2f} ± {std_stroke:.2f}",
        'p_value': p_value
    })
    
    # Print results
    print(f"{var}:")
    print(f"  Overall (N={df[var].notna().sum()}): {overall_mean:.2f} ± {overall_std:.2f}")
    print(f"  No Stroke (n={len(df_no_stroke)}): {mean_no_stroke:.2f} ± {std_no_stroke:.2f}")
    print(f"  Stroke (n={len(df_stroke)}): {mean_stroke:.2f} ± {std_stroke:.2f}")
    print(f"  p-value: {p_value:.10f} {'**' if p_value < 0.001 else ''}")
    print("")

# Convert to a DataFrame for easy viewing
df_results_continuous = pd.DataFrame(results_continuous)
print(df_results_continuous)

--- Analysis of Continuous Variables ---
Format: Mean ± Standard Deviation

age:
  Overall (N=5109): 43.23 ± 22.61
  No Stroke (n=4861): 41.97 ± 22.29
  Stroke (n=249): 67.73 ± 12.73
  p-value: 0.0000000000 **

avg_glucose_level:
  Overall (N=5109): 106.14 ± 45.29
  No Stroke (n=4861): 104.80 ± 43.85
  Stroke (n=249): 132.54 ± 61.92
  p-value: 0.0000000000 **

bmi:
  Overall (N=5109): 28.89 ± 7.70
  No Stroke (n=4861): 28.83 ± 7.78
  Stroke (n=249): 30.22 ± 5.83
  p-value: 0.0053619571 

            Variable         Overall       No_Stroke          Stroke  \
0                age   43.23 ± 22.61   41.97 ± 22.29   67.73 ± 12.73   
1  avg_glucose_level  106.14 ± 45.29  104.80 ± 43.85  132.54 ± 61.92   
2                bmi    28.89 ± 7.70    28.83 ± 7.78    30.22 ± 5.83   

        p_value  
0  7.030778e-71  
1  2.767811e-21  
2  5.361957e-03  


## chi test for categorical variables

In [6]:
# List of categorical variables to analyze
categorical_vars = ['hypertension', 'heart_disease', 'gender', 'ever_married', 
                    'work_type', 'Residence_type', 'smoking_status']

print("\n--- Analysis of Categorical Variables ---")
print("Format: Count (%)")
print("")

results_categorical = []

for var in categorical_vars:
    # Create a contingency table (cross-tabulation)
    ctab = pd.crosstab(df[var], df['stroke'], margins=True, margins_name='Total')
    ctab_percent = pd.crosstab(df[var], df['stroke'], normalize='columns') * 100
    
    # Perform Chi-square test of independence
    # contingency_table without the 'Total' margins
    ctab_for_test = pd.crosstab(df[var], df['stroke'])
    chi2, p_value, dof, expected = chi2_contingency(ctab_for_test)
    
    # Store results for each category level
    for level in ctab.index:
        if level != 'Total': # Skip the 'Total' row
            # Get counts for this level
            count_no_stroke = ctab_for_test.loc[level, 0]
            count_stroke = ctab_for_test.loc[level, 1]
            count_total = count_no_stroke + count_stroke
            
            # Get percentages for this level
            percent_no_stroke = ctab_percent.loc[level, 0]
            percent_stroke = ctab_percent.loc[level, 1]
            percent_total = (count_total / len(df)) * 100
            
            # Store results
            results_categorical.append({
                'Variable': var,
                'Level': level,
                'Overall_Count': count_total,
                'Overall_Percent': f"{percent_total:.1f}%",
                'No_Stroke_Count': count_no_stroke,
                'No_Stroke_Percent': f"{percent_no_stroke:.1f}%",
                'Stroke_Count': count_stroke,
                'Stroke_Percent': f"{percent_stroke:.1f}%",
                'p_value': p_value # same for all levels of this variable
            })
    
    # Print summary for the variable
    print(f"{var}:")
    print(ctab)
    print(f"Chi-square p-value: {p_value:.10f} {'**' if p_value < 0.001 else ''}")
    print("")

# Convert to a DataFrame for easy viewing
df_results_categorical = pd.DataFrame(results_categorical)
print(df_results_categorical)


--- Analysis of Categorical Variables ---
Format: Count (%)

hypertension:
stroke           0    1  Total
hypertension                  
0             4428  183   4611
1              432   66    498
Total         4860  249   5109
Chi-square p-value: 0.0000000000 **

heart_disease:
stroke            0    1  Total
heart_disease                  
0              4631  202   4833
1               229   47    276
Total          4860  249   5109
Chi-square p-value: 0.0000000000 **

gender:
stroke     0    1  Total
gender                  
Female  2853  141   2994
Male    2007  108   2115
Total   4860  249   5109
Chi-square p-value: 0.5598277581 

ever_married:
stroke           0    1  Total
ever_married                  
No            1727   29   1756
Yes           3133  220   3353
Total         4860  249   5109
Chi-square p-value: 0.0000000000 **

work_type:
stroke            0    1  Total
work_type                      
Govt_job        624   33    657
Never_worked     22    0     22
Private

In [7]:
import pandas as pd

# Assuming 'data' is the DataFrame containing the dataset

categorical_vars = ['hypertension', 'heart_disease', 'gender', 'ever_married', 
                    'work_type', 'Residence_type', 'smoking_status']

# Creating a function to calculate value counts and percentages
def calculate_counts_and_percentages(data, var):
    counts = data[var].value_counts()
    percentages = data[var].value_counts(normalize=True) * 100
    return counts, percentages

# Creating a function to calculate counts and percentages for stroke (0 and 1)
def calculate_stroke_counts_and_percentages(data, var):
    stroke_counts = data.groupby([var, 'stroke']).size().unstack(fill_value=0)
    stroke_percentages = stroke_counts.div(stroke_counts.sum(axis=1), axis=0) * 100
    return stroke_counts, stroke_percentages

# Looping through categorical variables to print the counts and percentages
for var in categorical_vars:
    print(f"--- {var} ---")
    
    # Value counts and percentages for the variable
    counts, percentages = calculate_counts_and_percentages(df, var)
    print("Counts:")
    print(counts)
    print("\nPercentages:")
    print(percentages)
    
    # Counts and percentages of stroke for each category
    stroke_counts, stroke_percentages = calculate_stroke_counts_and_percentages(df, var)
    print("\nStroke Counts (0: No Stroke, 1: Stroke):")
    print(stroke_counts)
    print("\nStroke Percentages (0: No Stroke, 1: Stroke):")
    print(stroke_percentages)
    
    print("\n" + "-"*50)


--- hypertension ---
Counts:
hypertension
0    4611
1     498
Name: count, dtype: int64

Percentages:
hypertension
0    90.252496
1     9.747504
Name: proportion, dtype: float64

Stroke Counts (0: No Stroke, 1: Stroke):
stroke           0    1
hypertension           
0             4428  183
1              432   66

Stroke Percentages (0: No Stroke, 1: Stroke):
stroke                0          1
hypertension                      
0             96.031230   3.968770
1             86.746988  13.253012

--------------------------------------------------
--- heart_disease ---
Counts:
heart_disease
0    4833
1     276
Name: count, dtype: int64

Percentages:
heart_disease
0    94.597769
1     5.402231
Name: proportion, dtype: float64

Stroke Counts (0: No Stroke, 1: Stroke):
stroke            0    1
heart_disease           
0              4631  202
1               229   47

Stroke Percentages (0: No Stroke, 1: Stroke):
stroke                 0          1
heart_disease                      
0  

/tmp/ipykernel_38/2595459981.py:16: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  stroke_counts = data.groupby([var, 'stroke']).size().unstack(fill_value=0)
/tmp/ipykernel_38/2595459981.py:16: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  stroke_counts = data.groupby([var, 'stroke']).size().unstack(fill_value=0)
/tmp/ipykernel_38/2595459981.py:16: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  stroke_counts = data.groupby